In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("EVENT.CSV")

In [3]:
df

,event_id,user_id,event_name,event_description,event_type,location,event_image,start_date,end_date,created_at,updated_at
0,1,348,Event 1,This is description for event 1,Sports,Delhi,event_1.jpg,05-09-2023 01:55,06-09-2023 01:55,31-08-2023 01:55,20-09-2023 01:55
1,2,626,Event 2,This is description for event 2,Technical,Mumbai,event_2.jpg,31-08-2023 11:46,31-08-2023 11:46,30-08-2023 11:46,06-09-2023 11:46
2,3,934,Event 3,This is description for event 3,Seminar,Bangalore,event_3.jpg,20-07-2024 10:45,23-07-2024 10:45,15-07-2024 10:45,19-07-2024 10:45
3,4,899,Event 4,This is description for event 4,Seminar,Bangalore,event_4.jpg,07-08-2023 17:01,09-08-2023 17:01,01-08-2023 17:01,31-08-2023 17:01
4,5,506,Event 5,This is description for event 5,Workshop,Delhi,event_5.jpg,11-08-2024 02:09,14-08-2024 02:09,07-08-2024 02:09,19-08-2024 02:09
...,...,...,...,...,...,...,...,...,...,...,...
3995,3996,538,Event 3996,This is description for event 3996,Workshop,Delhi,event_3996.jpg,03-07-2025 18:09,04-07-2025 18:09,29-06-2025 18:09,02-07-2025 18:09
3996,3997,892,Event 3997,This is description for event 3997,Seminar,Bangalore,event_3997.jpg,29-04-2024 07:03,01-05-2024 07:03,22-04-2024 07:03,27-04-2024 07:03
3997,3998,106,Event 3998,This is description for event 3998,Technical,Mumbai,event_3998.jpg,27-03-2025 09:34,30-03-2025 09:34,25-03-2025 09:34,16-04-2025 09:34
3998,3999,298,Event 3999,This is description for event 3999,Seminar,Mumbai,event_3999.jpg,07-07-2024 22:11,10-07-2024 22:11,02-07-2024 22:11,28-07-2024 22:11


In [4]:
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [5]:
date_cols = ['start_date', 'end_date', 'created_at', 'updated_at']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%d-%m-%Y %H:%M', errors='coerce')


In [6]:
df[date_cols].isnull().sum()


start_date    0
end_date      0
created_at    0
updated_at    0
dtype: int64

In [7]:
df['event_type'] = df['event_type'].str.title()
df['location'] = df['location'].str.title()
df['event_name'] = df['event_name'].str.title()


In [8]:
invalid_images = df[~df['event_image'].str.endswith(('.jpg', '.png', '.jpeg'))]


In [9]:
duplicate_event_id = df['event_id'].duplicated().sum()
duplicate_user_id = df['user_id'].duplicated().sum()


In [10]:
invalid_event_dates = df[df['end_date'] < df['start_date']]


In [11]:
invalid_update_dates = df[df['updated_at'] < df['created_at']]


In [12]:
total_rows = len(df)

report = pd.DataFrame({
    "Column_Name": df.columns,
    "Data_Type": df.dtypes.values,
    "Total_Records": total_rows,
    "Missing_Count": df.isnull().sum().values,
    "Missing_%": ((df.isnull().sum() / total_rows) * 100).round(2).values,
    "Duplicate_Count": [df[col].duplicated().sum() for col in df.columns],
    "Unique_Values": df.nunique().values
})

report["Invalid_End_Before_Start"] = report["Column_Name"].apply(
    lambda x: len(invalid_event_dates) if x in ["start_date","end_date"] else 0
)

report["Invalid_Update_Before_Create"] = report["Column_Name"].apply(
    lambda x: len(invalid_update_dates) if x in ["created_at","updated_at"] else 0
)


In [13]:
report

,Column_Name,Data_Type,Total_Records,Missing_Count,Missing_%,Duplicate_Count,Unique_Values,Invalid_End_Before_Start,Invalid_Update_Before_Create
0,event_id,int64,4000,0,0.0,0,4000,0,0
1,user_id,int64,4000,0,0.0,3019,981,0,0
2,event_name,str,4000,0,0.0,0,4000,0,0
3,event_description,str,4000,0,0.0,0,4000,0,0
4,event_type,str,4000,0,0.0,3995,5,0,0
5,location,str,4000,0,0.0,3995,5,0,0
6,event_image,str,4000,0,0.0,0,4000,0,0
7,start_date,datetime64[us],4000,0,0.0,1502,2498,0,0
8,end_date,datetime64[us],4000,0,0.0,1501,2499,0,0
9,created_at,datetime64[us],4000,0,0.0,1501,2499,0,0


In [15]:
df.to_csv("cleaned_event_data.csv", index=False)
report.to_csv("event_data_quality_report.csv", index=False)
